# CSC8101- Engineering for AI - 2025 Spark Coursework

## Coursework overview

### Inputs

- **NYC Taxi Trips dataset** - list of recorded taxi trips, each with several characteristics, namely: distance, number of passengers, origin zone, destination zone and trip cost (total amount charged to customer).
- **NYC Zones dataset** - list of zones wherein trips can originate/terminate.

### Tasks

1. Data cleaning
  1. Remove "0 distance" and 'no passengers' records.
  2. Remove outlier records.
2. Add new columns
  1. Join with zones dataset
  2. Compute the unit profitability of each trip
3. Zone summarisation and ranking
  1. Summarise trip data per zone
  2. Obtain the top 10 ranks according to:
    1. The total trip volume
    2. Their average profitabilitiy
    3. The total passenger volume
4. Record the total and task-specific execution times for each dataset size and format.


#### Code structure and implementation

  All the tasks are included in one code cell.

The following codeblock is used to mount the google colab instance to your google drive, in which we will be accessing the data provided in the course.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
pip install findspark

Below are the imports of various packages required.
The code line with spark session is creating or getting an existing SparkSession named "Taxi Data Analysis" with customized memory configurations for both the executor and driver processes. It allocates 8 GB of memory to each, which could help optimize performance, especially for memory-intensive jobs like data processing or machine learning tasks.

In [3]:
## global imports
import pyspark.sql as ps
from pyspark.sql.functions import col,  abs, percentile_approx, round, sum, count, mean, desc, broadcast
import pandas as pd
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.storagelevel import StorageLevel
import time

spark = SparkSession.builder \
    .appName("Taxi Data Analysis").config("spark.executor.memory", "8g").config("spark.driver.memory", "8g").getOrCreate()


## Task 0 - Read data

The following code block does the below:

- Reads the 'zones' dataset into variable 'zone_names'
- Defines the `init_trips` function that allows you to read the 'trips' dataset (from the DBFS FileStore) given the dataset size ('S' to 'XXL') as function argument
- Defines the `pipeline` function, called in Task 4 to measure the execution time of the entire data processing pipeline
- Shows you how to call the `init_trips` function and display dataset characteristics (number of rows, schema)

In [4]:
# Load zone names dataset - (much faster to read small file from git than dbfs)
zones_file_url = 'https://raw.githubusercontent.com/NewcastleComputingScience/csc8101-coursework/main/02-assignment-spark/taxi_zone_names.csv'
zone_names = spark.createDataFrame(pd.read_csv(zones_file_url))

# Function to load trips dataset by selected dataset size
def init_trips(size='S'):

    files = {
        'S'  : ['2021_07'],
        'M'  : ['2021'],
        'L'  : ['2020_21'],
        'XL' : ['1_6_2019', '7_12_2019'],
        'XXL': ['1_6_2019', '7_12_2019', '2020_21']
    }

    # validate input dataset size
    if size not in files.keys():
        print("Invalid input dataset size. Must be one of {}".format(list(files.keys())))
        return None

    filenames = list(map(lambda s: f'/content/drive/MyDrive/CSC8101_Data/tripdata_{s}.parquet', files[size]))
    trips_df = spark.read.parquet(filenames[0])

    for name in filenames[1:]:
        trips_df = trips_df.union(spark.read.parquet(name))
    print(
    """
    Trips dataset loaded!
    ---
      Size: {s}
      Tables loaded: {ds}
      Number of trips (dataset rows): {tc:,}
    """.format(s = size, ds = filenames, tc = trips_df.count()))

    return trips_df

# helper function to print dataset row count
def print_count(df):
    print("Row count: {t:,}".format(t = df.count()))

def pipeline(trips_df, with_task_12 = False, zones_df = zone_names):
    # Do not edit
    #---

    ## Task 1.1
    _trips_11 = t11_remove_zeros(trips_df)

    ## Task 1.2
    if with_task_12:
        _trips_12 = t12_remove_outliers(_trips_11)
    else:
        _trips_12 = _trips_11

    ## Task 2.1
    _trips_21 = t21_join_zones(_trips_12, zones_df = zone_names)

    ## Task 2.2
    _trips_22 = t22_calc_profit(_trips_21)

    ## Task 3.1
    _graph = t31_summarise_trips(_trips_22)

    ## Task 3.2
    _zones = t32_summarise_zones_pairs(_graph)

    _top10_trips     = t32_top10_trips(_zones)
    _top10_profit    = t32_top10_profit(_zones)
    _top10_passenger = t32_top10_passenger(_zones)

    return([_top10_trips, _top10_profit, _top10_passenger])

In [5]:
# CHANGE the value of argument 'size' to record the pipeline execution times for increasing dataset sizes
SIZE = 'M'

# Load trips dataset
trips = init_trips(SIZE)

# uncomment line only for small datasets
# trips.take(1)


    Trips dataset loaded!
    ---
      Size: M
      Tables loaded: ['/content/drive/MyDrive/CSC8101_Data/tripdata_2021.parquet']
      Number of trips (dataset rows): 15,571,166
    


In [ ]:
print_count(trips)

Row count: 2,898,033


In [ ]:
# dataset schemas
trips.printSchema()

root
 |-- index: long (nullable = true)
 |-- VendorID: double (nullable = true)
 |-- tpep_pickup_datetime: string (nullable = true)
 |-- tpep_dropoff_datetime: string (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- cab_type: string (nullable = true)
 |-- lpep_pickup_datetime: string (nullable = true)
 |-- lpep_dropoff_datetime: string (nullable = true)
 |--

In [ ]:
display(trips[['PULocationID', 'DOLocationID', 'trip_distance', 'passenger_count', 'total_amount']].take(5))

[Row(PULocationID=90, DOLocationID=68, trip_distance=0.8, passenger_count=1.0, total_amount=8.8),
 Row(PULocationID=113, DOLocationID=90, trip_distance=0.9, passenger_count=1.0, total_amount=8.8),
 Row(PULocationID=88, DOLocationID=232, trip_distance=2.8, passenger_count=1.0, total_amount=13.8),
 Row(PULocationID=79, DOLocationID=249, trip_distance=1.4, passenger_count=1.0, total_amount=12.3),
 Row(PULocationID=142, DOLocationID=238, trip_distance=2.0, passenger_count=0.0, total_amount=12.3)]

In [ ]:
zone_names.printSchema()

root
 |-- LocationID: long (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [ ]:
display(zone_names.take(5))

[Row(LocationID=1, Borough='EWR', Zone='Newark Airport', service_zone='EWR'),
 Row(LocationID=2, Borough='Queens', Zone='Jamaica Bay', service_zone='Boro Zone'),
 Row(LocationID=3, Borough='Bronx', Zone='Allerton/Pelham Gardens', service_zone='Boro Zone'),
 Row(LocationID=4, Borough='Manhattan', Zone='Alphabet City', service_zone='Yellow Zone'),
 Row(LocationID=5, Borough='Staten Island', Zone='Arden Heights', service_zone='Boro Zone')]

## Task 1 - Filter rows

**Input:** trips dataset

### Task 1.1 - Remove "0 distance" and 'no passengers' records

Remove dataset rows that represent invalid trips:

- Trips where `trip_distance == 0` (no distance travelled)
- Trips where `passenger_count == 0` and `total_amount == 0` (we want to retain records where `total_amount` > 0 - these may be significant as the taxi may have carried some parcel, for example)

Altogether, a record is removed if it satisfies the following conditions:

`trip_distance == 0` or `(passenger_count == 0` and `total_amount == 0)`.

**Recommended:** Select only the relevant dataset columns for this and subsequent tasks: `['PULocationID', 'DOLocationID', 'trip_distance', 'passenger_count', 'total_amount')]`

### Task 1.2 - Remove outliers using the modified z-score

Despite having removed spurious "zero passengers" trips in task 1.1, columns `total_amount` and `trip_distance` contain additional outlier values that must be identified and removed.

To identify and remove outliers, you will use the modified [z-score](https://en.wikipedia.org/wiki/Standard_score) method.
The modified z-score uses the median and [Median Absolute Deviation](https://en.wikipedia.org/wiki/Median_absolute_deviation) (MAD), instead of the mean and standard deviation, to determine how far an observation (indexed by i) is from the mean:

$$z_i = \frac{x_i - \mathit{median}(\mathbf{x})}{\mathbf{MAD}},$$

where x represents the input vector, xi is an element of x and zi is its corresponding z-score. In turn, the MAD formula is:

$$\mathbf{MAD} = 1.483 * \mathit{median}(\big\lvert x_i - \mathit{median}(\mathbf{x})\big\rvert).$$

Observations with **high** (absolute) z-score are considered outlier observations. A score is considered **high** if its __absolute z-score__ is larger than a threshold T = 3.5:

$$\big\lvert z_i \big\rvert > 3.5.$$

where T represents the number of unit standard deviations beyond which a score is considered an outlier ([wiki](https://en.wikipedia.org/wiki/68%E2%80%9395%E2%80%9399.7_rule)).

This process is repeated twice, once for each of the columns `total_amount` and `trip_distance` (in any order).

**Important:** Use the surrogate function [`percentile_approx`](https://spark.apache.org/docs/3.2.0/api/python/reference/api/pyspark.sql.functions.percentile_approx.html?highlight=percentile#pyspark.sql.functions.percentile_approx) to estimate the median (calculating the median values for a column is expensive as it cannot be parallelised efficiently).

## Task 2 - Compute new columns

### Task 2.1 - Zone names

Obtain the **start** and **end** zone names of each trip by joining the `trips` and `zone_names` datasets (i.e. by using the `zone_names` dataset as lookup table).

**Note:** The columns containing the start and end zone ids of each trip are named `PULocationID` and `DOLocationID`, respectively.

### Task 2.2 - Unit profitability

Compute the column `unit_profitabilty = total_amount / trip_distance`.

## Task 3: Rank zones by traffic, passenger volume and profitability

### 3.1 - Summarise interzonal travel

Build a graph data structure of zone-to-zone traffic, representing aggregated data about trips between any two zones. The graph will have one node for each zone and one edge connecting each pair of zones. In addition, edges contain aggregate information about all trips between those zones.

For example, zones Z1 and Z2 are connected by *two* edges: edge Z1 --> Z2 carries aggregate data about all trips that originated in Z1 and ended in Z2, and edge Z2 --> Z2 carries aggregate data about all trips that originated in Z2 and ended in Z1.

The aggregate information of interzonal travel must include the following data:

- `average_unit_profit` - the average unit profitability (calculated as `mean(unit_profitabilty)`).
- `trips_count` -- the total number of recorded trips.
- `total_passengers` -- the total number of passenger across all trips (sum of `passenger_count`).

This graph can be represented as a new dataframe, with schema:

\[`PULocationID`, `DOLocationID`, `average_unit_profit`, `trips_count`, `total_passengers` \]

__hint__: the `groupby()` operator produces a `pyspark.sql.GroupedData` structure. You can then calculate multiple aggregations from this using `pyspark.sql.GroupedData.agg()`:
- https://spark.apache.org/docs/3.2.0/api/python/reference/pyspark.pandas/api/pyspark.pandas.DataFrame.groupby.html
- https://spark.apache.org/docs/3.2.0/api/python/reference/api/pyspark.sql.GroupedData.agg.html

### Task 3.2 - Obtain top-10 zones

For each of the following measures, report the top-10 zones _using their plain names you dereferenced in the previous step, not the codes_. Note that this requires ranking the nodes in different orders. Specifically, you need to calculate the following further aggregations:

- the **total** number of trips originating from Z. This is simply the sum of `trips_count` over all outgoing edges for Z, i.e., edges of the form Z -> \*
- the **average** profitability of a zone. This is the average of all `average_unit_profit` over all *outgoing* edges from Z.
- The **total** passenger volume measured as the **sum** of `total_passengers` carried in trips that originate from Z

Below is the functions written for each task

In [5]:
# Solution implementation to task 1.1
def t11_remove_zeros(df):
    # input: trips dataset
    df = df.filter(
    (col("trip_distance") > 0) & #trip distance=0
    ~((col("passenger_count") == 0) & (col("total_amount") == 0))
    )
    return df

# Solution implementation to task 1.2
def t12_remove_outliers(df):
    columns = ["total_amount", "trip_distance"]
    for column in columns:

      # Calculate Median
      median_value = df.select(percentile_approx(column, 0.5, 10).alias("median")).collect()[0]["median"]

      # Calculate Absolute Deviations from Median
      df = df.withColumn("mad", abs(col(column) - median_value))

      # Calculate MAD
      mad_value = df.select(percentile_approx("mad", 0.5, 100).alias("mad")).collect()[0]["mad"]
      mad_value *= 1.483 #formula mentioned in text block


      # Calculate Z-Score
      df = df.withColumn("z_score", (col(column) - median_value) / mad_value)

      # Thresholding
      df = df.filter(abs(col("z_score")) <= 3.5).drop("z_score", "mad")

    return df


# Solution implementation to task 2.1

def t21_join_zones(df, zones_df = zone_names):

      # Join trips data with zone names on Pickup Location ID, using a broadcast join for optimization
      # Broadcast small dataset (zones) to speed up the join
      # Left join to keep all trip records even if no matching zone is found

    df = df \
        .join(broadcast(zone_names.alias("pu")), df.PULocationID == col("pu.LocationID"), "left") \
        .join(broadcast(zone_names.alias("do")), df.DOLocationID == col("do.LocationID"), "left") \
        .select(
            df["*"],
            col("pu.Zone").alias("start_zone"),
            col("do.Zone").alias("end_zone") # Select the original columns plus human-readable start and end zone
        )
    return df

# Solution implementation to task 2.2
def t22_calc_profit(df):
    # input: output of task 2.1
    # unit profitability
    df = df.withColumn("unit_profitability",
                       round(col("total_amount") / col("trip_distance"),2)
                       )

    return df

#Solution implementation to task 3.1

def t31_summarise_trips(df):

    # Group by Pickup and Drop-off Location IDs to analyze trip patterns

    # Calculate the average profitability per trip and round to 2 decimal places
    df_zoned = df.groupBy("PULocationID", "DOLocationID").agg(
        round(mean("unit_profitability"),2).alias("average_unit_profit"),

        # Count the total number of trips for each pickup-dropoff pair
        count("*").alias("trips_count"),

        # Sum up the total number of passengers across all trips
        sum("passenger_count").cast("int").alias("total_passengers")
    )
    return df_zoned

# Your solution to task 3.2 goes HERE (implement each of the functions below)
def t32_summarise_zones_pairs(df, zones_df = zone_names):

    # Summarize by Pickup Location (Origin Zone)
    # Calculate the total number of trips for each pickup location
    df_zones = df.groupBy("PULocationID").agg(

        # Compute the average unit profitability per pickup location, rounded to 2 decimal places
        sum("trips_count").alias("total_trips"),
        round(mean("average_unit_profit"), 2).alias("avg_profitability"),

        # Sum up the total number of passengers for each pickup location
        sum("total_passengers").alias("total_passengers")
    )

    # Join with Zone Names to replace ID with actual names
    df_zones = df_zones.join(zones_df, df_zones.PULocationID == zones_df.LocationID, "left") \
                       .select("Zone", "total_trips", "avg_profitability", "total_passengers")


    return df_zones

# Top 10 ranked zones by traffic (trip volume)

def t32_top10_trips(df_zones):
    # input: output of task 3.2
    # Sort the zones in descending order based on the total number of trips
    # Select only the top 10 zones with the highest trip volume

    top_trips = df_zones.orderBy(desc("total_trips")).limit(10)

    return top_trips

# Top 10 ranked zones by profit
def t32_top10_profit(df_zones):
    # input: output of task 3.2
    # Sort the zones in descending order based on the profit
    # Select only the top 10 zones with the highest profit
    top_profits=df_zones.orderBy(desc("avg_profitability")).limit(10)
    return top_profits

# Top 10 ranked zones by passenger volume
def t32_top10_passenger(df_zones):
    # input: output of task 3.2
    # Sort the zones in descending order based on the passenger volume
    # Select only the top 10 zones with the highest passenger volume
    top_passenger=df_zones.orderBy(desc("total_passengers")).limit(10)
    return top_passenger

## Task 4 - Record the pipeline's execution time

Record the execution time of:

1. the whole pipeline
2. the whole pipeline except task 1.2

on the two tables below, for all dataset sizes: `'S'`, `'M'`, `'L'`, `'XL'`, `'XXL'`.

Analyse the resulting execution times and comment on the effect of dataset size, dataset format and task complexity (with and without task 1.2) on pipeline performance.

In [7]:
# Define dataset sizes
dataset_sizes = ["S","M","L","XL","XXL"]

# Initialize lists to store results for each dataset size
results_with_12 = {'rows (M)': [], 'execution time': [], 'sec / 1M records': []}
results_without_12 = {'rows (M)': [], 'execution time': [], 'sec / 1M records': []}


In [9]:
# run and record the resulting execution time shown by databricks (on the cell footer)

# IMPORTANT: this function calls all task functions in order of occurrence. For this code to run without errors, you have to load into memory all of the previous task-specific functions, even if you haven't implemented these yet.
# pipeline(trips, with_task_12 = WITH_TASK_12)


# Iterate over dataset sizes
for size in dataset_sizes:
    print(f"Running pipeline for dataset size: {size}")

    # Load trips dataset
    trips = init_trips(size)
    # Row count in millions
    row_count = trips.count() / 1_000_000

    ## Run pipeline WITH Task 1.2
    # Record the start time
    start_time = time.time()
    # Run the pipeline with task 12 enabled
    pipeline(trips, with_task_12=True)
    # Record the end time
    end_time = time.time()
    # Calculate the time taken for task 12
    time_with_12 = end_time - start_time
    # Calculate the time taken per million rows, if row_count is greater than 0
    sec_per_million_12 = time_with_12 / row_count if row_count > 0 else 0


    ## Run pipeline WITHOUT Task 1.2
    start_time = time.time()
    pipeline(trips, with_task_12=False)
    end_time = time.time()
    time_without_12 = end_time - start_time
    sec_per_million_without_12 = time_without_12 / row_count if row_count > 0 else 0

    # Append the results for each metric in the respective dictionary
    results_with_12['rows (M)'].append(row_count)
    results_with_12['execution time'].append(time_with_12)
    results_with_12['sec / 1M records'].append(sec_per_million_12)

    results_without_12['rows (M)'].append(row_count)
    results_without_12['execution time'].append(time_without_12)
    results_without_12['sec / 1M records'].append(sec_per_million_without_12)
    print(f"Done Running pipeline for dataset size: {size}")


print(results_with_12)
print(results_without_12)

Running pipeline for dataset size: S

    Trips dataset loaded!
    ---
      Size: S
      Tables loaded: ['/content/drive/MyDrive/CSC8101_Data/tripdata_2021_07.parquet']
      Number of trips (dataset rows): 2,898,033
    
Done Running pipeline for dataset size: S
Running pipeline for dataset size: M

    Trips dataset loaded!
    ---
      Size: M
      Tables loaded: ['/content/drive/MyDrive/CSC8101_Data/tripdata_2021.parquet']
      Number of trips (dataset rows): 15,571,166
    
Done Running pipeline for dataset size: M
Running pipeline for dataset size: L

    Trips dataset loaded!
    ---
      Size: L
      Tables loaded: ['/content/drive/MyDrive/CSC8101_Data/tripdata_2020_21.parquet']
      Number of trips (dataset rows): 41,953,716
    
Done Running pipeline for dataset size: L
Running pipeline for dataset size: XL

    Trips dataset loaded!
    ---
      Size: XL
      Tables loaded: ['/content/drive/MyDrive/CSC8101_Data/tripdata_1_6_2019.parquet', '/content/drive/MyDrive/C

In [10]:
# Display tables in Databricks-style

columns = ["metric", "S","M","L","XL","XXL"]

# use pandas df to show in table
df_with_12 = pd.DataFrame(results_with_12,index=dataset_sizes).T
print("Table 1: Pipeline Performance WITH Task 1.2")
df_with_12


Table 1: Pipeline Performance WITH Task 1.2


,S,M,L,XL,XXL
rows (M),2.898033,15.571166,41.953716,90.443069,132.396785
execution time,6.499210,38.299039,137.133034,240.078655,174.613074
sec / 1M records,2.242628,2.459613,3.268674,2.654473,1.318862


In [11]:
df_without_12 = pd.DataFrame(results_without_12, index=dataset_sizes).T
print("\nTable 2: Pipeline Performance WITHOUT Task 1.2")
df_without_12


Table 2: Pipeline Performance WITHOUT Task 1.2


,S,M,L,XL,XXL
rows (M),2.898033,15.571166,41.953716,90.443069,132.396785
execution time,0.263074,0.206305,0.263374,0.218410,0.249118
sec / 1M records,0.090777,0.013249,0.006278,0.002415,0.001882


###### CONCLUSION

As the dataset size increases (from S to XXL), the overall execution time increases, which is expected since larger datasets require more processing. For example, in Table 1 (with Task 1.2), the execution time rises from 7.98 seconds for the smallest dataset (S) to 299.98 seconds for the largest dataset (XXL).

**With Task 1.2 (Table 1):** Task 1.2 significantly increases execution times,
especially for smaller datasets. The execution time is much higher compared to when Task 1.2 is excluded. For instance, the execution time for dataset size S with Task 1.2 is 7.98 seconds, whereas without Task 1.2, it’s just 0.22 seconds (Table 2).


**Without Task 1.2 (Table 2):** The pipeline runs significantly faster, with execution times dropping dramatically. The time per million records also decreases rapidly for larger datasets. This suggests that Task 1.2 is a bottleneck for smaller datasets, and removing it optimizes performance.